# Per-drug comparison: our DE vs colleague's DESeq2

For each of the 16 drug contexts we run a self-contained block:

1. **Load** both pipelines' outputs for that one drug.
   - **Ours**: `FDR_matrices/<drug>_FDRs.csv` + `PosteriorMeanMatrices/PosteriorMean_matrix_<drug>.csv`.
     PMs are on the natural-log scale; we divide by ln 2 to express them in log2.
   - **Theirs**: every `<PERT>_target_in_<drug>_25pct.csv` from
     `/large_storage/gilbertlab/tfair/Set{1,2}_all_FINAL/DESeq2/`. Multiple guide-pair
     files can target the same bare gene symbol; we union their significant calls and
     average their `log2FoldChange` per gene.
2. **Scatter** every (target_gene, measured_gene) cell — x = ours log2FC, y = theirs log2FC —
   and annotate the Pearson and Spearman correlations across all cells.
3. **Jaccard vs |log2FC| threshold** curve at the fixed FDR/padj cutoff: a gene is
   significant for (target_gene, drug, τ) iff
   - ours: FDR < `FDR_THR` ∧ |PM/ln 2| > τ
   - theirs: padj ≤ `PADJ_THR` ∧ |log2FoldChange| > τ
   For each (target_gene, drug, τ) we restrict both sig sets to the **per-pert** tested
   intersection — the genes both pipelines actually tested for that specific contrast.

After the loop we save the long-format Jaccard + per-drug correlation tables and produce
a cross-drug heatmap summarizing the sweep.

Memory is bounded to one drug at a time: per-drug data is dropped before the next drug.

In [1]:
from __future__ import annotations
import os, re, gc
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import anndata as ad
import scipy.sparse as sp
from tqdm.auto import tqdm
from scipy import stats

/home/beraslan/miniconda/envs/py312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
PROJ        = Path('/home/beraslan/Projects/ChemoGeneticScreens')
OUR_FDR_DIR = PROJ / 'FDR_matrices'
OUR_PM_DIR  = PROJ / 'PosteriorMeanMatrices'
OUR_RAW_DIR = Path('/processed_datasets/VCI/ChemoGenetic_H1_Basak')   # for per-pert cell counts
SET1_DIR    = Path('/large_storage/gilbertlab/tfair/Set1_all_FINAL/DESeq2')
SET2_DIR    = Path('/large_storage/gilbertlab/tfair/Set2_all_FINAL/DESeq2')
OUT_DIR     = OUR_FDR_DIR / 'jaccard_outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

FILTER     = '25pct'   # colleague's KD-quantile filter
FDR_THR    = 0.01      # our FDR cutoff
PADJ_THR   = 0.01      # their padj cutoff
CONTRAST   = 'target'

# Log-spaced |log2FC| sweep — gives the smooth curves the 06 plots used.
# τ=0 is kept for the FDR-only baseline (the log-x plots simply drop it).
N_THRESHOLDS    = 30
THRESHOLDS_LOG2 = np.concatenate([[0.0],
                                   np.logspace(np.log10(0.01), np.log10(5.0), N_THRESHOLDS)])
LN2 = float(np.log(2))
THRESHOLDS_NAT  = THRESHOLDS_LOG2 * LN2

# Cell-count bins for stratifying per-pert curves (06-style).
N_CELL_BINS   = [2, 50, 100, 200, 400, 800, 1600, 3000, 1_000_000]
N_CELL_LABELS = ['2-50', '50-100', '100-200', '200-400',
                 '400-800', '800-1600', '1600-3000', '3000+']

# Round-number log ticks for the τ axis (ported from 06).
_LOG_TICKS = np.array([
    0.005, 0.007,
    0.01, 0.015, 0.02, 0.03, 0.04, 0.05, 0.07,
    0.10, 0.15, 0.20, 0.30, 0.40, 0.50, 0.70,
    1.00, 1.50, 2.00, 3.00, 4.00, 5.00, 7.00,
    10.0, 15.0, 20.0,
])

def _fmt_tick(x: float) -> str:
    if x >= 10: return f'{x:.0f}'
    if x >= 1:  return f'{x:.1f}'
    return f'{x:g}'

DRUG_SPECS = [
    # batch 1 (Set 1)
    {'set': 'batch1', 'their_dir': SET1_DIR, 'their_drug': 'AR.A014418',           'our_drug': 'AR-A014418'},
    {'set': 'batch1', 'their_dir': SET1_DIR, 'their_drug': 'AZD4573',              'our_drug': 'AZD4573'},
    {'set': 'batch1', 'their_dir': SET1_DIR, 'their_drug': 'CHIR.98014',           'our_drug': 'CHIR-98014'},
    {'set': 'batch1', 'their_dir': SET1_DIR, 'their_drug': 'DMSO',                 'our_drug': 'DMSO_round2'},
    {'set': 'batch1', 'their_dir': SET1_DIR, 'their_drug': 'Lexibulin',            'our_drug': 'Lexibulin'},
    {'set': 'batch1', 'their_dir': SET1_DIR, 'their_drug': 'PP121',                'our_drug': 'PP121'},
    {'set': 'batch1', 'their_dir': SET1_DIR, 'their_drug': 'Romidepsin',           'our_drug': 'Romidepsin'},
    {'set': 'batch1', 'their_dir': SET1_DIR, 'their_drug': 'Stattic',              'our_drug': 'Stattic'},
    # batch 2 (Set 2)
    {'set': 'batch2', 'their_dir': SET2_DIR, 'their_drug': 'Bisindolylmaleimide.I','our_drug': 'Bisindolylmaleimide-I'},
    {'set': 'batch2', 'their_dir': SET2_DIR, 'their_drug': 'DG.172',               'our_drug': 'DG-172'},
    {'set': 'batch2', 'their_dir': SET2_DIR, 'their_drug': 'DMSO',                 'our_drug': 'DMSO_round2_batch2'},
    {'set': 'batch2', 'their_dir': SET2_DIR, 'their_drug': 'JTE.607',              'our_drug': 'JTE-607'},
    {'set': 'batch2', 'their_dir': SET2_DIR, 'their_drug': 'LDN.193189',           'our_drug': 'LDN-193189'},
    {'set': 'batch2', 'their_dir': SET2_DIR, 'their_drug': 'LY2090314',            'our_drug': 'LY2090314'},
    {'set': 'batch2', 'their_dir': SET2_DIR, 'their_drug': 'NSC95397',             'our_drug': 'NSC95397'},
    {'set': 'batch2', 'their_dir': SET2_DIR, 'their_drug': 'VX.11e',               'our_drug': 'VX-11e'},
]

DESEQ2_COLNAMES = ['gene', 'baseMean', 'log2FoldChange', 'lfcSE', 'stat', 'pvalue', 'padj']
_SUFFIX_RE      = re.compile(r'_(P\d+(?:P\d+)?|ENST[\d.]+)_[AB]$')

def pert_to_gene(pert_id: str) -> str:
    return _SUFFIX_RE.sub('', pert_id.split('|')[0])

print(f'{len(DRUG_SPECS)} drugs, {len(THRESHOLDS_LOG2)} thresholds in [0, '
      f'{THRESHOLDS_LOG2[-1]:.2f}], {len(N_CELL_LABELS)} cell-count bins '
      f'({N_CELL_BINS[0]}..{N_CELL_BINS[-2]:,}+), FILTER={FILTER!r}, '
      f'FDR<{FDR_THR}, padj≤{PADJ_THR}')

16 drugs, 31 thresholds in [0, 5.00], 8 cell-count bins (2..3,000+), FILTER='25pct', FDR<0.01, padj≤0.01


## Per-drug helpers

- `load_one_drug(spec)` — reads both pipelines' outputs for one drug into memory and
  returns a dict of (log2FC matrices, sig sets per τ, tested sets per pert).
- `compute_jaccard_for_drug(data, spec)` — produces a long-format Jaccard DataFrame at
  every τ for every shared target, using the per-pert tested intersection.
- `compute_pooled_corr(data)` — Pearson + Spearman over all (target × gene) cells.
- `plot_drug(spec, data, jacc_drug, corr)` — the per-drug 1×2 figure: log2FC scatter on
  the left, Jaccard-vs-τ curve on the right.

In [3]:
def load_one_drug(spec: dict) -> dict:
    # Returns a dict with:
    #   their_l2fc, their_sig, their_tested, their_npairs,
    #   our_l2fc,   our_sig,   our_tested,
    #   n_cells_per_pert         — {target_gene: int} from our source h5ad (backed read of obs).
    #   mean_expr_ctrl_per_gene  — pd.Series indexed by gene, mean log1p expression in NT cells.
    their_dir, their_drug, our_drug = spec['their_dir'], spec['their_drug'], spec['our_drug']
    suffix = f'_{CONTRAST}_in_{their_drug}_{FILTER}.csv'
    files  = sorted(f for f in os.listdir(their_dir) if f.endswith(suffix))

    # ---- colleague side ----
    their_sig_acc:    dict[str, dict[float, set[str]]] = defaultdict(lambda: {float(t): set() for t in THRESHOLDS_LOG2})
    their_tested_acc: dict[str, set[str]]              = defaultdict(set)
    their_l2fc_acc:   dict[str, list[pd.Series]]       = defaultdict(list)

    for fname in tqdm(files, desc=f'load theirs {our_drug}', leave=False):
        gene = pert_to_gene(fname[:-len(suffix)])
        df = pd.read_csv(their_dir / fname,
                         header=0, names=DESEQ2_COLNAMES,
                         usecols=[0, 2, 6], index_col=0)
        idx     = df.index.astype(str).values
        padj    = df['padj'].values
        l2fc    = df['log2FoldChange'].values
        padj_ok = ~np.isnan(padj) & (padj <= PADJ_THR)
        fc_ok   = ~np.isnan(l2fc)

        their_tested_acc[gene].update(idx)
        their_l2fc_acc[gene].append(df['log2FoldChange'].astype(np.float32))
        per_tau = their_sig_acc[gene]
        for tau in THRESHOLDS_LOG2:
            mask = padj_ok & fc_ok & (np.abs(l2fc) > tau)
            per_tau[float(tau)].update(idx[mask])

    their_sig    = {g: {t: frozenset(s) for t, s in d.items()} for g, d in their_sig_acc.items()}
    their_tested = {g: frozenset(s) for g, s in their_tested_acc.items()}
    their_npairs = {g: len(sl) for g, sl in their_l2fc_acc.items()}

    their_l2fc_rows = {}
    for g, sl in their_l2fc_acc.items():
        their_l2fc_rows[g] = sl[0] if len(sl) == 1 else pd.concat(sl, axis=1).mean(axis=1)
    their_l2fc = pd.DataFrame(their_l2fc_rows).T.astype(np.float32)

    # ---- our side: FDR + PM matrices ----
    fdr_fp = OUR_FDR_DIR / f'{our_drug}_FDRs.csv'
    pm_fp  = OUR_PM_DIR  / f'PosteriorMean_matrix_{our_drug}.csv'
    if not fdr_fp.exists() or not pm_fp.exists():
        raise FileNotFoundError(f'missing matrices for {our_drug}')
    fdr_df = pd.read_csv(fdr_fp, index_col=0)
    pm_df  = pd.read_csv(pm_fp,  index_col=0)
    common_perts = fdr_df.index.intersection(pm_df.index)
    common_genes = fdr_df.columns.intersection(pm_df.columns)
    fdr_df = fdr_df.loc[common_perts, common_genes]
    pm_df  = pm_df.loc[common_perts,  common_genes]

    fdr_arr = fdr_df.values
    pm_arr  = pm_df.values
    abs_pm  = np.abs(pm_arr)
    fdr_ok  = ~np.isnan(fdr_arr) & (fdr_arr < FDR_THR)
    pm_ok   = ~np.isnan(pm_arr)
    cols    = common_genes.astype(str).values

    our_sig:    dict[str, dict[float, frozenset[str]]] = {}
    our_tested: dict[str, frozenset[str]]              = {}
    for i, tg in enumerate(common_perts.astype(str)):
        per_tau = {}
        for tau_l2, tau_nat in zip(THRESHOLDS_LOG2, THRESHOLDS_NAT):
            mask = fdr_ok[i] & pm_ok[i] & (abs_pm[i] > tau_nat)
            per_tau[float(tau_l2)] = frozenset(cols[mask].tolist())
        our_sig[tg]    = per_tau
        our_tested[tg] = frozenset(cols[~np.isnan(fdr_arr[i])].tolist())

    our_l2fc = (pm_df / LN2).astype(np.float32)

    # ---- per-pert cell counts + per-gene mean log1p expression in NT cells ----
    # Both come from the source h5ad. Backed read of obs is cheap; materialising the
    # NT-cell subset of X is a few hundred MB of sparse → fine on this box.
    src_h5 = OUR_RAW_DIR / our_drug / f'{our_drug}.h5ad'
    n_cells_per_pert: dict[str, int] = {}
    mean_expr_ctrl_per_gene: pd.Series = pd.Series(dtype=np.float32)
    if src_h5.exists():
        a = ad.read_h5ad(src_h5, backed='r')
        try:
            tg_series = a.obs['target_gene'].astype(str)
            n_cells_per_pert = tg_series.value_counts().to_dict()
            is_nt = tg_series.isin(['non-targeting', 'non_targeting']).values
            if is_nt.any():
                a_nt = a[is_nt].to_memory()
                X_nt = a_nt.X
                if sp.issparse(X_nt):
                    mean_ctrl = np.asarray(X_nt.mean(axis=0)).ravel()
                else:
                    mean_ctrl = np.asarray(X_nt).mean(axis=0)
                mean_expr_ctrl_per_gene = pd.Series(
                    mean_ctrl.astype(np.float32),
                    index=a_nt.var_names.astype(str),
                    name='mean_expr_ctrl',
                )
                del a_nt
            else:
                print(f'  [warn] no NT cells found in {src_h5.name}')
        finally:
            try: a.file.close()
            except Exception: pass
        del a
    else:
        print(f'  [warn] source h5ad not found, cell counts + ctrl means disabled: {src_h5}')

    return {
        'their_l2fc':              their_l2fc,
        'their_sig':               their_sig,
        'their_tested':            their_tested,
        'their_npairs':            their_npairs,
        'our_l2fc':                our_l2fc,
        'our_sig':                 our_sig,
        'our_tested':              our_tested,
        'n_cells_per_pert':        n_cells_per_pert,
        'mean_expr_ctrl_per_gene': mean_expr_ctrl_per_gene,
    }

In [4]:
def compute_jaccard_for_drug(data: dict, spec: dict) -> pd.DataFrame:
    """Per (target_gene, τ) Jaccard, restricted to the per-pert tested intersection."""
    rows = []
    shared = sorted(set(data['their_sig']) & set(data['our_sig']))
    for tg in shared:
        common_pp = data['their_tested'].get(tg, frozenset()) & data['our_tested'].get(tg, frozenset())
        if not common_pp:
            for tau in THRESHOLDS_LOG2:
                rows.append({'batch': spec['set'], 'drug_ours': spec['our_drug'],
                             'drug_theirs': spec['their_drug'], 'target_gene': tg,
                             'tau_log2': tau,
                             'n_their_sig': 0, 'n_our_sig': 0, 'n_intersect': 0,
                             'n_union': 0, 'jaccard': np.nan,
                             'n_tested_intersect': 0,
                             'n_guide_pairs_theirs': data['their_npairs'].get(tg, 0)})
            continue
        for tau in THRESHOLDS_LOG2:
            a = data['their_sig'][tg][tau] & common_pp
            b = data['our_sig'][tg][tau]   & common_pp
            inter = a & b
            union = a | b
            rows.append({
                'batch':              spec['set'],
                'drug_ours':          spec['our_drug'],
                'drug_theirs':        spec['their_drug'],
                'target_gene':        tg,
                'tau_log2':           tau,
                'n_their_sig':        len(a),
                'n_our_sig':          len(b),
                'n_intersect':        len(inter),
                'n_union':            len(union),
                'jaccard':            (len(inter) / len(union)) if union else np.nan,
                'n_tested_intersect': len(common_pp),
                'n_guide_pairs_theirs': data['their_npairs'].get(tg, 0),
            })
    return pd.DataFrame(rows)


def compute_pooled_corr(data: dict) -> dict:
    """Pearson + Spearman across all (target × gene) cells with finite log2FC on both sides."""
    A = data['their_l2fc']
    B = data['our_l2fc']
    common_perts = A.index.intersection(B.index)
    common_genes = A.columns.intersection(B.columns)
    a = A.loc[common_perts, common_genes].values.ravel()
    b = B.loc[common_perts, common_genes].values.ravel()
    ok = np.isfinite(a) & np.isfinite(b)
    n = int(ok.sum())
    if n < 10 or np.std(a[ok]) == 0 or np.std(b[ok]) == 0:
        return {'n': n, 'pearson': float('nan'), 'spearman': float('nan')}
    return {
        'n':        n,
        'pearson':  float(stats.pearsonr(a[ok],  b[ok])[0]),
        'spearman': float(stats.spearmanr(a[ok], b[ok])[0]),
    }

In [ ]:
def _apply_log_tau_ticks(ax, lo: float, hi: float):
    ax.set_xscale('log')
    ax.set_xlim(lo * 0.95, hi * 1.05)
    xt = _LOG_TICKS[(_LOG_TICKS >= lo) & (_LOG_TICKS <= hi)]
    ax.set_xticks(xt)
    ax.set_xticklabels([_fmt_tick(t) for t in xt], rotation=0)
    ax.minorticks_off()


def plot_drug(spec: dict, data: dict, jacc_drug: pd.DataFrame, corr: dict,
              *, scatter_lim: float = 5.0, ma_ylim: float = 5.0):
    # 4×3 figure:
    #   row 0: [ scatter | Jaccard vs τ           | Intersection size vs τ          ]
    #   row 1: [ (blank) | Basak's sig count vs τ | Tyler's sig count vs τ          ]
    #   row 2: [ (blank) | Basak's MA (all cells) | Tyler's MA (all cells)          ]
    #   row 3: [ (blank) | Basak's MA (sig only)  | Tyler's MA (sig only)           ]
    fig, axes = plt.subplots(4, 3, figsize=(20, 22))

    # ---------- (0,0): log2FC scatter ----------
    A = data['their_l2fc']
    B = data['our_l2fc']
    common_perts = A.index.intersection(B.index)
    common_genes = A.columns.intersection(B.columns)
    a = A.loc[common_perts, common_genes].values.ravel()
    b = B.loc[common_perts, common_genes].values.ravel()
    ok = np.isfinite(a) & np.isfinite(b)
    a, b = a[ok], b[ok]
    L = scatter_lim

    ax = axes[0, 0]
    ax.hexbin(b, a, gridsize=70, mincnt=1, cmap='viridis', bins='log',
              extent=(-L, L, -L, L))
    ax.plot([-L, L], [-L, L], ls='--', color='crimson', lw=0.8, label='y = x')
    ax.axhline(0, color='gray', lw=0.4); ax.axvline(0, color='gray', lw=0.4)
    ax.set_xlim(-L, L); ax.set_ylim(-L, L)
    ax.set_xlabel(r"Basak's  $\log_2\,\mathrm{FC}$")
    ax.set_ylabel(r"Tyler's  $\log_2\,\mathrm{FC}$")
    ax.set_title(f'log2FC scatter — ρ={corr["pearson"]:.3f}  ρ_s={corr["spearman"]:.3f}'
                 f'  (n={corr["n"]:,})', fontsize=10)
    ax.legend(loc='upper left', fontsize=8)

    # ---------- stratify perts by cell count ----------
    n_series = pd.Series(data['n_cells_per_pert'], name='n_cells')
    pert_bin = pd.cut(n_series, bins=N_CELL_BINS, labels=N_CELL_LABELS,
                       include_lowest=True)
    pert_to_bin = pert_bin.dropna().astype(str).to_dict()
    jdf = jacc_drug.copy()
    jdf['n_cells_bin'] = jdf['target_gene'].map(pert_to_bin)
    jdf = jdf[jdf['n_cells_bin'].notna()]
    plot_df = jdf[jdf['tau_log2'] > 0]

    palette = sns.color_palette('viridis', n_colors=len(N_CELL_LABELS))
    x_lo = float(plot_df['tau_log2'].min()) if len(plot_df) else 0.01
    x_hi = float(plot_df['tau_log2'].max()) if len(plot_df) else 5.0

    def _strat_curves(ax, *, value_col, ylabel, title,
                       logy=False, dropna_col=None,
                       legend_loc='upper right', ylim=None):
        src = plot_df.dropna(subset=[dropna_col]) if dropna_col else plot_df
        for color, lab in zip(palette, N_CELL_LABELS):
            sub = src[src['n_cells_bin'] == lab]
            if sub.empty:
                continue
            g = sub.groupby('tau_log2')[value_col]
            med = g.median()
            lo  = g.quantile(0.25)
            hi  = g.quantile(0.75)
            n   = sub['target_gene'].nunique()
            ax.plot(med.index.values, med.values, color=color, lw=1.8,
                    label=f'{lab}  (n={n})')
            ax.fill_between(med.index.values, lo.values, hi.values,
                             color=color, alpha=0.15, linewidth=0)
        _apply_log_tau_ticks(ax, x_lo, x_hi)
        if logy:
            ax.set_yscale('symlog', linthresh=1)
        if ylim is not None:
            ax.set_ylim(*ylim)
        ax.set_xlabel(r'$|\log_2\,\mathrm{FC}|$ threshold $\tau$  (log scale)')
        ax.set_ylabel(ylabel)
        ax.set_title(title, fontsize=10)
        ax.legend(loc=legend_loc, fontsize=7, frameon=True, title='# cells (Basak)')
        ax.grid(alpha=0.3, which='major')

    _strat_curves(axes[0, 1], value_col='jaccard',
                  ylabel='Jaccard  (median ± IQR)',
                  title=f"Jaccard: Basak's vs Tyler's sig sets, vs τ   (FDR<{FDR_THR}, padj≤{PADJ_THR})",
                  logy=False, dropna_col='jaccard',
                  legend_loc='upper left', ylim=(0, 1.0))
    _strat_curves(axes[0, 2], value_col='n_intersect',
                  ylabel='# genes called sig by both, per target  (median ± IQR)',
                  title="Intersection size: Basak's ∩ Tyler's, vs τ",
                  logy=True)

    axes[1, 0].set_axis_off()
    _strat_curves(axes[1, 1], value_col='n_our_sig',
                  ylabel="# Basak's sig genes per target  (median ± IQR)",
                  title=f"Basak's sig-gene count vs τ   (FDR<{FDR_THR} ∧ |log2FC|>τ)",
                  logy=True)
    _strat_curves(axes[1, 2], value_col='n_their_sig',
                  ylabel="# Tyler's sig genes per target  (median ± IQR)",
                  title=f"Tyler's sig-gene count vs τ   (padj≤{PADJ_THR} ∧ |log2FC|>τ)",
                  logy=True)

    # ---------- MA plot helpers ----------
    mean_ctrl = data.get('mean_expr_ctrl_per_gene', pd.Series(dtype=np.float32))

    def _ma_setup(ax, l2fc_df):
        if mean_ctrl is None or mean_ctrl.empty:
            ax.text(0.5, 0.5, '(no NT mean expression available)',
                    ha='center', va='center', transform=ax.transAxes, fontsize=10)
            ax.set_axis_off(); return None
        common = l2fc_df.columns.intersection(mean_ctrl.index)
        if len(common) == 0:
            ax.text(0.5, 0.5, '(no shared genes between l2fc & ctrl-mean)',
                    ha='center', va='center', transform=ax.transAxes, fontsize=10)
            ax.set_axis_off(); return None
        mean_x = mean_ctrl.loc[common].values.astype(np.float32)
        Y      = l2fc_df[common].values.astype(np.float32)
        return mean_x, Y, common

    def _ma_render(ax, mean_x, x_flat, y_flat, keep, label, title_extra):
        if not keep.any():
            ax.text(0.5, 0.5, '(no points to plot)',
                    ha='center', va='center', transform=ax.transAxes, fontsize=10)
            ax.set_axis_off(); return
        x_lo = float(np.nanmin(mean_x)); x_hi = float(np.nanmax(mean_x))
        ax.hexbin(x_flat[keep], y_flat[keep], gridsize=80, mincnt=1,
                  cmap='viridis', bins='log',
                  extent=(x_lo, x_hi, -ma_ylim, ma_ylim))
        ax.axhline(0, color='crimson', lw=0.6, ls='--')
        ax.set_xlim(x_lo, x_hi)
        ax.set_ylim(-ma_ylim, ma_ylim)
        ax.set_xlabel('mean log1p expression of gene in NT cells of this drug')
        ax.set_ylabel(f"{label}'s  $\\log_2\\,\\mathrm{{FC}}$")
        ax.set_title(f"{label}'s MA plot — {title_extra}", fontsize=10)
        ax.grid(alpha=0.3)

    def _ma_panel_all(ax, l2fc_df, label):
        setup = _ma_setup(ax, l2fc_df)
        if setup is None: return
        mean_x, Y, _ = setup
        n_perts, n_genes = Y.shape
        x_flat = np.tile(mean_x, n_perts)
        y_flat = Y.ravel()
        keep = np.isfinite(x_flat) & np.isfinite(y_flat)
        _ma_render(ax, mean_x, x_flat, y_flat, keep, label,
                   f'all (target, gene) cells  ({n_perts:,} × {n_genes:,})')

    def _ma_panel_sig(ax, l2fc_df, sig_dict, label, threshold_str):
        setup = _ma_setup(ax, l2fc_df)
        if setup is None: return
        mean_x, Y, common = setup
        n_perts, n_genes = Y.shape
        gene_to_col = {g: i for i, g in enumerate(common.astype(str).tolist())}
        sig_mask = np.zeros_like(Y, dtype=bool)
        for row_i, tg in enumerate(l2fc_df.index.astype(str).tolist()):
            per_tau = sig_dict.get(tg)
            if per_tau is None: continue
            for g in per_tau.get(0.0, ()):
                ci = gene_to_col.get(g)
                if ci is not None: sig_mask[row_i, ci] = True
        x_flat = np.tile(mean_x, n_perts)
        y_flat = Y.ravel()
        keep   = sig_mask.ravel() & np.isfinite(x_flat) & np.isfinite(y_flat)
        n_sig  = int(keep.sum())
        _ma_render(ax, mean_x, x_flat, y_flat, keep, label,
                   f'significant only ({threshold_str}; n={n_sig:,})')

    axes[2, 0].set_axis_off()
    _ma_panel_all(axes[2, 1], data['our_l2fc'],   'Basak')
    _ma_panel_all(axes[2, 2], data['their_l2fc'], 'Tyler')

    axes[3, 0].set_axis_off()
    _ma_panel_sig(axes[3, 1], data['our_l2fc'],   data['our_sig'],
                  'Basak', f'FDR<{FDR_THR}')
    _ma_panel_sig(axes[3, 2], data['their_l2fc'], data['their_sig'],
                  'Tyler', f'padj≤{PADJ_THR}')

    fig.suptitle(f"{spec['our_drug']}  [{spec['set']}]   "
                 f"(Tyler's drug name: {spec['their_drug']})",
                 fontsize=12, y=1.00)
    fig.tight_layout()
    return fig


def plot_sig_counts_per_pert(spec: dict, jacc_drug: pd.DataFrame, data: dict,
                              *, n_cols: int = 6):
    """Second per-drug figure: grid of scatter panels, one per τ in THRESHOLDS_LOG2.
    Each panel: x = Basak's # sig genes per target, y = Tyler's # sig genes per target.
    Color encodes cell-count bin. Crimson dashed = y = x. Symlog axes (linthresh=1) so
    counts of 0 sit on the axis cleanly.

    'Significant' uses the joint criterion at that τ:
        Basak  : FDR     < FDR_THR  ∧ |log2FC| > τ
        Tyler  : padj    ≤ PADJ_THR ∧ |log2FC| > τ
    """
    # Per-target cell-count bin (same scheme as plot_drug).
    n_series = pd.Series(data['n_cells_per_pert'], name='n_cells')
    pert_bin = pd.cut(n_series, bins=N_CELL_BINS, labels=N_CELL_LABELS, include_lowest=True)
    pert_to_bin = pert_bin.dropna().astype(str).to_dict()
    jdf = jacc_drug.copy()
    jdf['n_cells_bin'] = jdf['target_gene'].map(pert_to_bin)
    jdf = jdf[jdf['n_cells_bin'].notna()]

    taus = sorted(jdf['tau_log2'].unique())
    n_taus = len(taus)
    n_rows = (n_taus + n_cols - 1) // n_cols

    palette = sns.color_palette('viridis', n_colors=len(N_CELL_LABELS))
    color_map = dict(zip(N_CELL_LABELS, palette))

    overall_max = max(int(jdf['n_our_sig'].max()),
                      int(jdf['n_their_sig'].max()), 1)
    lim = overall_max * 1.10

    fig, axes = plt.subplots(n_rows, n_cols,
                              figsize=(3.0 * n_cols, 3.0 * n_rows),
                              sharex=True, sharey=True)
    axes_flat = np.atleast_1d(axes).flatten()

    for i, tau in enumerate(taus):
        ax = axes_flat[i]
        sub = jdf[jdf['tau_log2'] == tau]
        for lab in N_CELL_LABELS:
            bin_sub = sub[sub['n_cells_bin'] == lab]
            if bin_sub.empty:
                continue
            ax.scatter(bin_sub['n_our_sig'], bin_sub['n_their_sig'],
                       s=5, alpha=0.5, color=color_map[lab], edgecolors='none')
        ax.plot([0, lim], [0, lim], ls='--', color='crimson', lw=0.7)
        ax.set_xscale('symlog', linthresh=1)
        ax.set_yscale('symlog', linthresh=1)
        ax.set_xlim(-0.5, lim)
        ax.set_ylim(-0.5, lim)
        ax.set_title(f'τ = {tau:.3g}   (n={len(sub):,} perts)', fontsize=9)
        ax.grid(alpha=0.25, which='major')

    for i in range(n_taus, len(axes_flat)):
        axes_flat[i].set_axis_off()

    fig.supxlabel("# Basak's sig genes per target   (symlog)", fontsize=11)
    fig.supylabel("# Tyler's sig genes per target   (symlog)", fontsize=11)

    handles = [plt.Line2D([0], [0], marker='o', color='w',
                           markerfacecolor=color_map[lab],
                           markeredgecolor='none', markersize=8, label=lab)
               for lab in N_CELL_LABELS]
    fig.legend(handles=handles, loc='lower right',
               bbox_to_anchor=(0.995, 0.005),
               title='# cells (Basak)', fontsize=9, ncol=1, frameon=True)

    fig.suptitle(
        f"{spec['our_drug']}  [{spec['set']}]   "
        f"Per-pert sig-gene count: Basak vs Tyler at each |log2FC| threshold τ\n"
        f"(FDR<{FDR_THR} for Basak, padj≤{PADJ_THR} for Tyler; one panel per τ; "
        f"crimson = y=x)",
        fontsize=12, y=1.00,
    )
    fig.tight_layout()
    return fig

## Per-drug loop

For each drug: load both sides → scatter + correlation → Jaccard curve. Per-drug data is
freed before the next drug. Aggregated Jaccard and correlation tables are accumulated for
the summary at the bottom of the notebook.

Heads-up on cost: the colleague's side requires reading ~2,350 CSVs per drug. Expect
~30–60 s per drug on this machine.

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages
from IPython.display import display

# Restrict the loop to a subset of drugs (set to None to run all 16).
RUN_DRUGS: list[str] | None = [
    'DMSO_round2',           # batch1 DMSO
    'DMSO_round2_batch2',    # batch2 DMSO
    'CHIR-98014',
    'LY2090314',
    'Stattic',
    'VX-11e',
]

specs_to_run = (DRUG_SPECS if RUN_DRUGS is None
                else [s for s in DRUG_SPECS if s['our_drug'] in set(RUN_DRUGS)])
missing = set(RUN_DRUGS or []) - {s['our_drug'] for s in DRUG_SPECS}
if missing:
    raise ValueError(f'RUN_DRUGS contains unknown drugs: {sorted(missing)}')
print(f'Running loop on {len(specs_to_run)} drug(s): '
      f'{[s["our_drug"] for s in specs_to_run]}')

all_jacc:   list[pd.DataFrame] = []
all_corrs:  list[dict]         = []
universe_rows: list[dict]      = []

pdf_path = OUT_DIR / 'per_drug_figures.pdf'
with PdfPages(pdf_path) as pdf:
    for spec in specs_to_run:
        print(f'\n=== {spec["our_drug"]}   [{spec["set"]}]   '
              f"(Tyler's: {spec['their_drug']}) ===", flush=True)

        data = load_one_drug(spec)

        n_shared_targets = len(set(data['their_sig']) & set(data['our_sig']))
        pp_intersects = [
            len(data['their_tested'][tg] & data['our_tested'][tg])
            for tg in (set(data['their_tested']) & set(data['our_tested']))
        ]
        pp_arr = np.asarray(pp_intersects) if pp_intersects else np.array([0])
        print(f'  shared targets:      {n_shared_targets:>5,d}')
        print(f'  per-pert tested ∩    min={pp_arr.min():>5,d}  med={int(np.median(pp_arr)):>5,d}  '
              f'max={pp_arr.max():>5,d}')

        jacc_drug = compute_jaccard_for_drug(data, spec)
        corr = compute_pooled_corr(data)
        print(f'  log2FC corr (pooled): ρ={corr["pearson"]:.3f}  '
              f'ρ_s={corr["spearman"]:.3f}  n={corr["n"]:,}')

        # Two pages per drug: main 4×3 figure, then the per-τ Basak-vs-Tyler-count grid.
        fig1 = plot_drug(spec, data, jacc_drug, corr)
        pdf.savefig(fig1, bbox_inches='tight')
        display(fig1)
        plt.close(fig1)

        fig2 = plot_sig_counts_per_pert(spec, jacc_drug, data)
        pdf.savefig(fig2, bbox_inches='tight')
        display(fig2)
        plt.close(fig2)

        all_jacc.append(jacc_drug)
        all_corrs.append({
            'batch':       spec['set'],
            'drug_ours':   spec['our_drug'],
            'drug_theirs': spec['their_drug'],
            **corr,
        })
        universe_rows.append({
            'batch':                spec['set'],
            'drug_ours':            spec['our_drug'],
            'drug_theirs':          spec['their_drug'],
            'n_ours_genes':         data['our_l2fc'].shape[1],
            'n_theirs_genes':       data['their_l2fc'].shape[1],
            'n_shared_targets':     n_shared_targets,
            'pp_intersect_min':     int(pp_arr.min()),
            'pp_intersect_median':  int(np.median(pp_arr)),
            'pp_intersect_max':     int(pp_arr.max()),
        })

        del data
        gc.collect()

    d = pdf.infodict()
    d['Title']    = "Per-drug DE comparison — Basak's vs Tyler's"
    d['Subject']  = f'Jaccard + intersection-size sweep over |log2FC| τ; FDR<{FDR_THR}, padj≤{PADJ_THR}, filter={FILTER}'
    d['Keywords'] = 'CRISPR; DESeq2; ashr; Jaccard; chemogenetic screen'

print(f'\nwrote {pdf_path}')

jacc_df     = pd.concat(all_jacc, ignore_index=True)
corr_df     = pd.DataFrame(all_corrs)
universe_df = pd.DataFrame(universe_rows)
print(f'=== loop done — {len(jacc_df):,} Jaccard rows across '
      f'{jacc_df["drug_ours"].nunique()} drugs ===')

## Cross-drug summary

Two compact views across all 16 drugs:

- **Pooled log2FC correlations + tested-gene universes** per drug.
- **Heatmap** of median Jaccard at every (drug, τ).

In [7]:
summary_df = (corr_df.merge(universe_df,
                            on=['batch', 'drug_ours', 'drug_theirs'],
                            how='left')
                     .sort_values(['batch', 'pearson'], ascending=[True, False])
                     .reset_index(drop=True))
print(summary_df.to_string(index=False))

 batch          drug_ours drug_theirs        n  pearson  spearman  n_ours_genes  n_theirs_genes  n_shared_targets  pp_intersect_min  pp_intersect_median  pp_intersect_max
batch1            Stattic     Stattic 21543237 0.473370  0.792221         18154           15043              2139              9928                 9943             12193
batch1        DMSO_round2        DMSO 21530145 0.443251  0.762664         18154           15043              2138              9928                 9943             12186
batch1         CHIR-98014  CHIR.98014 21461087 0.412078  0.742895         18154           15043              2132              9928                 9943             12193
batch2             VX-11e      VX.11e 21687075 0.461527  0.783500         18154           14928              2135             10021                10052             11923
batch2 DMSO_round2_batch2        DMSO 21676115 0.423595  0.758605         18154           14928              2134             10021              

In [ ]:
# Drug × τ heatmap of median Jaccard.  Too many τ columns to annotate cleanly →
# render as a smooth gradient with viridis colormap.
median_by_drug_tau = (jacc_df.dropna(subset=['jaccard'])
                      .groupby(['drug_ours', 'tau_log2'])['jaccard']
                      .median()
                      .unstack('tau_log2'))
# Use `specs_to_run` so the row order works whether you ran all drugs or a subset.
row_order = [s['our_drug'] for s in specs_to_run]
median_by_drug_tau = median_by_drug_tau.loc[row_order]

fig, ax = plt.subplots(figsize=(12, max(3, 0.5 * len(row_order) + 2)))
sns.heatmap(median_by_drug_tau, annot=False, cmap='viridis', vmin=0, vmax=1,
            cbar_kws={'label': 'median Jaccard'},
            ax=ax, linewidths=0, linecolor='white')
n_batch1 = sum(1 for s in specs_to_run if s['set'] == 'batch1')
if 0 < n_batch1 < len(specs_to_run):
    ax.axhline(n_batch1, color='white', lw=2)
ax.set_xlabel(r'$|\log_2\,\mathrm{FC}|$ threshold $\tau$')
ax.set_ylabel('')
xt = np.arange(len(median_by_drug_tau.columns))
xstep = max(1, len(xt) // 8)
ax.set_xticks(xt[::xstep] + 0.5)
ax.set_xticklabels([f'{c:.2f}' for c in median_by_drug_tau.columns[::xstep]], rotation=0)
ax.set_title(f'Median Jaccard per (drug, τ)  (FDR<{FDR_THR}, padj≤{PADJ_THR}, filter={FILTER})')
plt.tight_layout()
plt.savefig(OUT_DIR / 'jaccard_heatmap_drug_x_tau.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Drug × τ heatmap of median intersection size.  Smooth gradient (no annotations).
from matplotlib.colors import LogNorm

median_isect_by_drug_tau = (
    jacc_df.groupby(['drug_ours', 'tau_log2'])['n_intersect']
           .median()
           .unstack('tau_log2')
)
median_isect_by_drug_tau = median_isect_by_drug_tau.loc[row_order]

plot_vals = median_isect_by_drug_tau.copy() + 1   # avoid log(0)
fig, ax = plt.subplots(figsize=(12, max(3, 0.5 * len(row_order) + 2)))
sns.heatmap(plot_vals, annot=False,
            cmap='mako_r',
            norm=LogNorm(vmin=1, vmax=max(plot_vals.values.max(), 2)),
            cbar_kws={'label': 'median # intersect (+1, log color)'},
            ax=ax, linewidths=0, linecolor='white')
if 0 < n_batch1 < len(specs_to_run):
    ax.axhline(n_batch1, color='white', lw=2)
ax.set_xlabel(r'$|\log_2\,\mathrm{FC}|$ threshold $\tau$')
ax.set_ylabel('')
ax.set_xticks(xt[::xstep] + 0.5)
ax.set_xticklabels([f'{c:.2f}' for c in median_isect_by_drug_tau.columns[::xstep]], rotation=0)
ax.set_title(f'Median # intersecting sig genes per target  (FDR<{FDR_THR}, padj≤{PADJ_THR}, filter={FILTER})')
plt.tight_layout()
plt.savefig(OUT_DIR / 'intersect_heatmap_drug_x_tau.png', dpi=150, bbox_inches='tight')
plt.show()

## Save

In [ ]:
tag = f'FDR{FDR_THR}_padj{PADJ_THR}_{FILTER}_tau{"-".join(str(t) for t in THRESHOLDS_LOG2)}'
jacc_path    = OUT_DIR / f'jaccard_per_target_{tag}.csv'
summary_path = OUT_DIR / 'per_drug_summary.csv'
jacc_df.to_csv(jacc_path, index=False)
summary_df.to_csv(summary_path, index=False)
print(f'wrote {jacc_path}  ({len(jacc_df):,} rows)')
print(f'wrote {summary_path}  ({len(summary_df)} drugs)')